[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lalo/TorchCode/blob/master/solutions/42_batchnorm2d_solution.ipynb)

# 🟡 Solution: Implement BatchNorm2d

Reference solution for per-channel Batch Normalization over `(N, C, H, W)`, matching `nn.BatchNorm2d` in training and inference, including the unbiased running-variance update.

### 📐 Notation

| Symbol | Meaning |
|:---:|---|
| `N` | batch size (PyTorch's image convention) |
| `C` | channels |
| `H` | image height |
| `W` | image width |


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def my_batch_norm2d(
    x,
    gamma,
    beta,
    running_mean,
    running_var,
    eps=1e-5,
    momentum=0.1,
    training=True,
):
    """BatchNorm over (N, C, H, W): one statistic per channel, over batch and space."""
    dims = (0, 2, 3)
    if training:
        mean = x.mean(dim=dims)                         # (C,)
        var = x.var(dim=dims, unbiased=False)           # (C,) biased: used to normalize
        with torch.no_grad():
            unbiased_var = x.var(dim=dims, unbiased=True)   # count is N * H * W
            running_mean.mul_(1 - momentum).add_(momentum * mean)
            running_var.mul_(1 - momentum).add_(momentum * unbiased_var)
    else:
        mean, var = running_mean, running_var

    shape = (1, -1, 1, 1)                               # broadcast (C,) over N, H, W
    x_norm = (x - mean.view(shape)) / torch.sqrt(var.view(shape) + eps)
    return gamma.view(shape) * x_norm + beta.view(shape)


In [ ]:
# Verify against nn.BatchNorm2d
torch.manual_seed(0)
x = torch.randn(4, 3, 8, 8) * 3 + 1
bn = torch.nn.BatchNorm2d(3).train()
running_mean, running_var = torch.zeros(3), torch.ones(3)
out = my_batch_norm2d(x, torch.ones(3), torch.zeros(3), running_mean, running_var, training=True)
print("Train output matches:", torch.allclose(out, bn(x), atol=1e-4))
print("Running stats match: ", torch.allclose(running_mean, bn.running_mean), torch.allclose(running_var, bn.running_var))
bn.eval()
x2 = torch.randn(2, 3, 8, 8)
print("Eval output matches: ", torch.allclose(my_batch_norm2d(x2, torch.ones(3), torch.zeros(3), running_mean, running_var, training=False), bn(x2), atol=1e-4))


In [ ]:
from torch_judge import check
check('batchnorm2d')
